## Llamada a un endpoint falso

In [1]:
## Smoke-test: forzar un error de Binance (endpoint inválido)
from time import time
from panzer.rate_limit.binance_fixed import BinanceFixedWindowLimiter
from panzer.http.client import binance_public_get, BINANCE_SPOT_BASE_URL
from panzer.errors import BinanceAPIException

# Limiter muy permisivo (no queremos que bloquee nada aquí)
unsafe_limiter = BinanceFixedWindowLimiter(
    max_per_minute=10 ** 9,
    safety_ratio=1.0,
)

try:
    t0 = time()
    data, headers = binance_public_get(
        base_url=BINANCE_SPOT_BASE_URL,
        endpoint="/api/v3/this_does_not_exist",  # endpoint inventado
        params=None,
        limiter=unsafe_limiter,
        weight=1,
        timeout=5,
    )
    t1 = time()
    print(f"NO ERROR (raro). status OK, elapsed={t1 - t0:.3f}s, data={data}")
except BinanceAPIException as exc:
    print("BinanceAPIException capturada:")
    print("  status_code:", exc.status_code)
    if exc.error_payload:
        print("  code:", exc.error_payload.code)
        print("  msg:", exc.error_payload.msg)
except Exception as e:
    print("Excepción inesperada:", repr(e))


2025-11-23 12:50:51    ERROR [panzer.errors] BinanceAPIException raised: status=404 method=GET url=https://api.binance.com/api/v3/this_does_not_exist code=None msg=None


BinanceAPIException capturada:
  status_code: 404


## Inicializar cliente Panzer (SPOT) antes del stress test

In [2]:
from panzer.exchanges.binance.public import BinancePublicClient

client = BinancePublicClient(market="spot", safety_ratio=0.9)
print("Cliente BinancePublicClient(spot) inicializado.")

2025-11-23 12:50:52     INFO [panzer.binance.spot] Market=spot REQUEST_WEIGHT limit: RateLimit(rate_limit_type='REQUEST_WEIGHT', interval='MINUTE', interval_num=1, limit=6000)
2025-11-23 12:50:52     INFO [panzer.binance.spot] BinancePublicClient inicializado para market=spot, base_url=https://api.binance.com, safety_ratio=0.9


Cliente BinancePublicClient(spot) inicializado.


## FASE 1: Reventar /api/v3/exchangeInfo hasta obtener un 429 real

In [3]:
## FASE 1: Reventar /api/v3/exchangeInfo hasta obtener un 429 real
from time import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import json
import requests

BASE_URL = "https://api.binance.com"
ENDPOINT = "/api/v3/exchangeInfo"  # endpoint pesado

MAX_JOBS = 5000
MAX_WORKERS = 50

stop_event = threading.Event()


def parallel_worker(idx: int):
    if stop_event.is_set():
        return idx, None, None, None, None

    url = BASE_URL + ENDPOINT
    t0 = time()
    resp = requests.get(url, timeout=10)
    elapsed = time() - t0

    status = resp.status_code
    used_1m = resp.headers.get("x-mbx-used-weight-1m")

    if status != 200:
        stop_event.set()

    return idx, status, elapsed, used_1m, resp


print(f"Lanzando hasta {MAX_JOBS} jobs con {MAX_WORKERS} workers...")
t_start = time()

futures = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    for i in range(MAX_JOBS):
        if stop_event.is_set():
            break
        futures.append(pool.submit(parallel_worker, i))

print(f"Jobs lanzados: {len(futures)}. Esperando resultados...")

any_error = False
processed = 0

for fut in as_completed(futures):
    idx, status, elapsed, used_1m, resp = fut.result()
    if status is None:
        continue

    processed += 1
    print(f"[i={idx}] status={status} elapsed={elapsed:.3f}s used_weight_1m={used_1m}")

    if status != 200:
        any_error = True
        print("\n=== RESPUESTA NO 200 DETECTADA ===")
        print(f"Status: {status}")
        print("Headers:")
        print(resp.headers)

        try:
            body = resp.json()
        except json.JSONDecodeError:
            body = resp.text

        print("Body:")
        print(body)
        break

t_total = time() - t_start

if not any_error:
    print(
        f"\nFin del experimento sin respuestas != 200 "
        f"(processed={processed}, duración total={t_total:.1f}s)."
    )


Lanzando hasta 5000 jobs con 50 workers...
Jobs lanzados: 5000. Esperando resultados...
[i=27] status=200 elapsed=2.174s used_weight_1m=380
[i=42] status=200 elapsed=2.331s used_weight_1m=840
[i=26] status=200 elapsed=2.121s used_weight_1m=500
[i=41] status=200 elapsed=1.994s used_weight_1m=160
[i=25] status=200 elapsed=2.161s used_weight_1m=480
[i=24] status=200 elapsed=2.262s used_weight_1m=240
[i=39] status=200 elapsed=2.233s used_weight_1m=520
[i=40] status=200 elapsed=2.255s used_weight_1m=940
[i=23] status=200 elapsed=2.219s used_weight_1m=280
[i=299] status=200 elapsed=1.131s used_weight_1m=680
[i=298] status=200 elapsed=1.430s used_weight_1m=640
[i=297] status=200 elapsed=1.112s used_weight_1m=660
[i=296] status=200 elapsed=1.073s used_weight_1m=600
[i=295] status=200 elapsed=1.246s used_weight_1m=580
[i=294] status=200 elapsed=1.591s used_weight_1m=620
[i=293] status=200 elapsed=1.181s used_weight_1m=560
[i=292] status=200 elapsed=1.132s used_weight_1m=540
[i=291] status=200 e

## FASE 2: Probar Panzer después de haber consumido el rate limit

In [4]:
## FASE 2: Probar el limitador de Panzer tras el abuso
from time import time
from datetime import datetime
import requests

BASE_URL = "https://api.binance.com"
TIME_ENDPOINT = "/api/v3/time"

# 1) Estado crudo del peso justo después del stress test
t0 = time()
resp = requests.get(BASE_URL + TIME_ENDPOINT, timeout=5)
elapsed_raw = time() - t0
used_1m_raw = resp.headers.get("x-mbx-used-weight-1m")

print("== Estado justo después del stress test ==")
print(f"status={resp.status_code} elapsed={elapsed_raw:.3f}s "
      f"used_weight_1m={used_1m_raw}")

# 2) Varias llamadas get_time() vía Panzer
print("\nLanzando varias llamadas get_time() vía Panzer...")
N = 5

for i in range(N):
    t0 = time()
    exc = None
    server_time = None

    try:
        server_time = client.get_time(timeout=5)
    except Exception as e:
        exc = e

    t1 = time()
    elapsed = t1 - t0

    used_local = client._limiter.used_local
    server_used = client._limiter.last_server_used

    print(
        f"[{i}] {datetime.utcnow().isoformat(timespec='seconds')} "
        f"elapsed={elapsed:.3f}s | used_local={used_local} | "
        f"server_used={server_used} | exc={repr(exc)}"
    )


== Estado justo después del stress test ==
status=429 elapsed=0.257s used_weight_1m=6301

Lanzando varias llamadas get_time() vía Panzer...


2025-11-23 12:51:08    ERROR [panzer.errors] BinanceAPIException raised: status=429 method=GET url=https://api.binance.com/api/v3/time code=-1003 msg=Too much request weight used; current limit is 6000 request weight per 1 MINUTE. Please use WebSocket Streams for live updates to avoid polling the API.
/tmp/ipykernel_43190/3392756593.py:40: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  f"[{i}] {datetime.utcnow().isoformat(timespec='seconds')} "
2025-11-23 12:51:08  WARNING [panzer.binance_rate_limit] Límite de seguridad alcanzado (used_local=6302, weight=1, effective_limit=5400). Durmiendo 51.05 segundos.


[0] 2025-11-23T11:51:08 elapsed=0.258s | used_local=6302 | server_used=6302 | exc=BinanceAPIException('[429] GET https://api.binance.com/api/v3/time (code=-1003, msg=Too much request weight used; current limit is 6000 request weight per 1 MINUTE. Please use WebSocket Streams for live updates to avoid polling the API.)')
[1] 2025-11-23T11:52:00 elapsed=51.336s | used_local=1 | server_used=1 | exc=None
[2] 2025-11-23T11:52:00 elapsed=0.257s | used_local=2 | server_used=2 | exc=None
[3] 2025-11-23T11:52:00 elapsed=0.257s | used_local=3 | server_used=3 | exc=None
[4] 2025-11-23T11:52:01 elapsed=0.261s | used_local=4 | server_used=4 | exc=None
